# 3. Updating KItems with the SDK

In this tutorial we see how to update existing KItems.

### 3.1. Setting up

Before you run this tutorial: make sure to have access to a DSMS-instance of your interest, along with installation of this package, and have established access to the DSMS through DSMS-SDK (refer to [Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms))


Now let us import the needed classes and functions for this tutorial.

In [ ]:
from dsms import DSMS, KItem

Now source the environmental variables from an `.env` file and start the DSMS-session.

In [ ]:
import os
dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()

Now let us get the KItem we created in the [2nd tutorial: Creation of KItems](2_creation.ipynb)


In [ ]:
item = KItem(
    name="Specimen123",
    ktype_id=dsms.ktypes.Specimen,
    custom_properties={"Width": 0.5, "Length": 0.15},
)
dsms.add(item)
dsms.commit()
item

In [ ]:
item

### 3.2. Updating KItems

Now, we would like to update the properties of our KItem we created previously.

Depending on the schema of each property (see [DSMS KItem Schema](../dsms_kitem_schema.md)), `list` accumulation (`+=` or `-=`), e.g. for the `annotations`, `attachments`, `external_link`, etc. 

**NOTE**: using `append` or `extend` will not validate the pydantic model of the respective fields and hence will cause an error during committing. Hence, always use `+=`, `=` or `-=`.

Other properties which are not `list`-like can be simply set by attribute-assignment (e.g. `name`, `slug`, `ktype_id`, etc).

In [ ]:
item.name = "Specimen-123"
item.custom_properties.Width = 1
item.attachments += ["testfile.txt"]
item.annotations += ["https://w3id.org/pmd/co/Specimen"]
item.external_links += [
    {"url": "http://specimens.org", "label": "specimen-link"}
]
item.contacts += [{"name": "Specimen preparation", "email": "specimenpreparation@group.mail"}]

In [ ]:
dsms.add(item)
dsms.commit()

We can see now that the local system path of the attachment is changed to a simple file name, which means that the upload was successful. If not so, an error would have been thrown during the `commit`.

We can see the updates when we print the item:

In [ ]:
item

### 3.3. Updating access properties

Access control entries can be updated in the same way as other KItem properties.

In [ ]:
from dsms.knowledge.properties.access import KItemAccessProperties, Role

# Look up the current user to demonstrate role assignment
uname = dsms.config.username
if hasattr(uname, "get_secret_value"):
    uname = uname.get_secret_value()
current_user = dsms.users.by_username.get(uname)

item.access_properties = KItemAccessProperties(
    user_access=[{"user_id": current_user.id, "role": Role.CONTRIBUTOR}],
)
dsms.commit()

Furthermore we can also download the file we uploaded again:

In [ ]:
for file in item.attachments:
    download = file.download()

    print("\t\t\t Downloaded file:", file.name)
    print("|------------------------------------Beginning of file------------------------------------|")
    print(download)
    print("|---------------------------------------End of file---------------------------------------|\n\n")

In [ ]:
# Clean up the tutorial item
del dsms[item]
dsms.commit()